# 토스 경진대회 최종 제출 - AP + WLL 최적화

**홍익대 3학년 | 데이터사이언스 | AI 해커톤 준비 중**

---
## 핵심 전략 (슬라이드 기반)
1. **Resampling**: 5-fold CV + **Bootstrap 추정**
2. **Classification**: **p-value 기반 범주형 필터링** → LightGBM
3. **WLL 대응**: `scale_pos_weight=1.0` + **Isotonic Calibration**
4. **AP 최적화**: **OOF 기반 Rank 보정**
5. **Model Assessment**: CV로 **test error 추정**

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# 디렉토리 설정
notebook_dir = r'C:\Users\tkdwl\Desktop\토스 경진대회'
os.chdir(notebook_dir)

# 데이터 로드
df_train = pd.read_parquet('./train.parquet')
df_test = pd.read_parquet('./test.parquet')

print(f"Train: {df_train.shape} | Test: {df_test.shape}")

Train: (10704179, 119) | Test: (1527298, 119)


## 1. 결측 처리 (슬라이드: Resampling 전 데이터 정제)

In [2]:
target_col = 'clicked'
id_col = 'ID'
col_drop_threshold = 0.05

# 5% 이상 결측 제거
missing = pd.concat([df_train.isnull().mean(), df_test.isnull().mean()], axis=1).max(axis=1)
high_missing_cols = missing[missing >= col_drop_threshold].index.tolist()
high_missing_cols = [c for c in high_missing_cols if c not in [target_col, id_col]]

df_train = df_train.drop(columns=high_missing_cols)
df_test = df_test.drop(columns=high_missing_cols)

# 수치형/범주형 분리
num_cols = df_train.select_dtypes(include=np.number).columns.tolist()
num_cols = [c for c in num_cols if c != target_col]
cat_cols = df_train.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c not in [id_col]]

# 수치형 결측 행 제거
df_train = df_train.dropna(subset=num_cols)
df_test = df_test.dropna(subset=num_cols)

# 범주형 최빈값
from sklearn.impute import SimpleImputer
cat_imputer = SimpleImputer(strategy='most_frequent')
df_train[cat_cols] = cat_imputer.fit_transform(df_train[cat_cols])
df_test[cat_cols] = cat_imputer.transform(df_test[cat_cols])

print(f"결측 처리 완료")

결측 처리 완료


## 2. 범주형 p-value 필터링 (Classification 슬라이드)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import statsmodels.api as sm

# day_of_week를 사용하여 is_weekend 피처 먼저 생성 (삭제 전)
if 'day_of_week' in df_train.columns: # day_of_week 피처가 존재하는 경우
    df_train['is_weekend'] = df_train['day_of_week'].astype(str).str.contains('5|6', regex=True).astype(int) # 요일이 5(토) 또는 6(일)인 경우 1, 아니면 0
    df_test['is_weekend'] = df_test['day_of_week'].astype(str).str.contains('5|6', regex=True).astype(int)
# 주말 여부 피처 새로 생성

# 범주형 변수 목록 정의
cat_for_selection = ['gender', 'age_group', 'inventory_id', 'day_of_week'] # 범주형 변수 후보 목록
X_cat = df_train[cat_for_selection].astype(str) # 이들 변수를 str로 변환해 저장
y = df_train[target_col] # 예측 대상(종속변수, 타겟)

# One-hot (범주형 -> 숫자형)
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore') # 범주형 변수를 더미 변수로 변환 ex) gender -> gender_F, gender_M
X_ohe = ohe.fit_transform(X_cat)
feature_names = ohe.get_feature_names_out(cat_for_selection) # 인코딩된 컬럼 이름들이 담김

# Logistic + L1 + p-value
X_const = sm.add_constant(X_ohe)
logit = sm.Logit(y, X_const) # statsmodels의 Logit을 이용해 로지스틱 회기 수행
result = logit.fit_regularized(method='l1', alpha=0.001,maxiter=1000, disp=False) # L1 규제를 사용해 과적합 방지
# 이렇게 하면 변수 선택이 일부 자동으로 된다.

# 유의미한 피처 선택
p_vals = result.pvalues[1:] 
selected_features = feature_names[p_vals < 0.05] # p-value가 0.05 미만인 피처만 선택

print(f"선택된 범주형 피처: {len(selected_features)}개")

# 적용
ohe_sel = OneHotEncoder(sparse_output=False)
ohe_sel.categories_ = ohe.categories_ # 같은 카테고리 구조를 유지하면서 다시 인코딩

train_ohe = pd.DataFrame(ohe_sel.fit_transform(X_cat), columns=feature_names)[selected_features]
test_ohe = pd.DataFrame(ohe_sel.transform(df_test[cat_for_selection].astype(str)), columns=feature_names)[selected_features]
# 이렇게 만든 train_ohe, test_ohe를 학습/테스트셋에 병합

for col in selected_features:
    df_train[col] = train_ohe[col].values
    df_test[col] = test_ohe[col].values

df_train = df_train.drop(columns=cat_for_selection)
df_test = df_test.drop(columns=cat_for_selection)

c:\Users\tkdwl\anaconda3\envs\pytorch\Lib\site-packages\statsmodels\base\l1_solvers_common.py:71: ConvergenceWarning: QC check did not pass for 24 out of 36 parameters
Try increasing solver accuracy or number of iterations, decreasing alpha, or switch solvers
  warnings.warn(message, ConvergenceWarning)
c:\Users\tkdwl\anaconda3\envs\pytorch\Lib\site-packages\statsmodels\base\l1_solvers_common.py:144: ConvergenceWarning: Could not trim params automatically due to failed QC check. Trimming using trim_mode == 'size' will still work.
  warnings.warn(msg, ConvergenceWarning)


선택된 범주형 피처: 0개


## 3. 피처 엔지니어링

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 시간 피처
df_train['hour'] = pd.to_numeric(df_train['hour'], errors='coerce').fillna(12)
df_test['hour'] = pd.to_numeric(df_test['hour'], errors='coerce').fillna(12)

for df in [df_train, df_test]:
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    # is_weekend는 이미 셀 5에서 생성됨 (day_of_week 삭제 전)

# seq 피처
def process_seq(df, top_n=15):
    df['seq_length'] = df['seq'].str.split(',').str.len().fillna(0)
    df['seq_unique'] = df['seq'].str.split(',').apply(lambda x: len(set(x)) if isinstance(x, str) else 0)
    df['seq_diversity'] = df['seq_unique'] / (df['seq_length'] + 1e-8)
    items = [i.strip() for s in df['seq'].str.split(',') for i in s if isinstance(s, str)]
    top_items = pd.Series(items).value_counts().head(50).index
    for item in top_items[:top_n]:
        df[f'seq_has_{item}'] = df['seq'].str.contains(item, regex=False).fillna(0).astype(int)
    return df

df_train = process_seq(df_train)
df_test = process_seq(df_test)

# 클러스터링
cluster_cols = [c for c in num_cols if c in df_train.columns]
scaler = StandardScaler()
scaled_train = scaler.fit_transform(df_train[cluster_cols])
scaled_test = scaler.transform(df_test[cluster_cols])

kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
df_train['cluster'] = kmeans.fit_predict(scaled_train)
df_test['cluster'] = kmeans.predict(scaled_test)

df_train = pd.get_dummies(df_train, columns=['cluster'], prefix='cluster')
df_test = pd.get_dummies(df_test, columns=['cluster'], prefix='cluster')
df_test = df_test.reindex(columns=df_train.columns, fill_value=0)

# 수치형 스케일링
num_cols_final = [c for c in num_cols if c in df_train.columns]
df_train[num_cols_final] = scaler.fit_transform(df_train[num_cols_final])
df_test[num_cols_final] = scaler.transform(df_test[num_cols_final])

print(f"최종 피처 수: {df_train.shape[1] - 2}")

# 처리된 데이터 저장


## 4. 5-fold CV + LightGBM

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score, log_loss
from sklearn.isotonic import IsotonicRegression
import gc

feature_cols = [c for c in df_train.columns if c not in [target_col, id_col]]
X = df_train[feature_cols]
y = df_train[target_col]

# WLL: 50:50 가중 → scale_pos_weight=1.0
params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.03,
    'num_leaves': 128,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'verbose': -1,
    'seed': 42
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(df_test))
ap_scores = []

for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y)):
    X_trn, X_val = X.iloc[trn_idx], X.iloc[val_idx]
    y_trn, y_val = y.iloc[trn_idx], y.iloc[val_idx]

    lgb_train = lgb.Dataset(X_trn, y_trn)
    lgb_valid = lgb.Dataset(X_val, y_val, reference=lgb_train)

    model = lgb.train(
        params,
        lgb_train,
        num_boost_round=10000,
        valid_sets=[lgb_valid],
        early_stopping_rounds=300,
        verbose_eval=1000
    )

    val_pred = model.predict(X_val)
    oof_preds[val_idx] = val_pred

    # AP 계산
    ap = average_precision_score(y_val, val_pred)
    ap_scores.append(ap)
    print(f"Fold {fold+1} AP: {ap:.5f}")

    test_preds += model.predict(df_test[feature_cols]) / skf.n_splits

print(f"\nCV AP: {np.mean(ap_scores):.5f} ± {np.std(ap_scores):.5f}")

In [ ]:
# 토스 경진대회 최종 제출 - AP + WLL 최적화
#
# **홍익대 3학년 | 데이터사이언스 | AI 해커톤 준비 중**
#
# ---
# ## 핵심 전략 (슬라이드 기반)
# 1. **Resampling**: 5-fold CV + **Bootstrap 추정**
# 2. **Classification**: **p-value 기반 범주형 필터링** → LightGBM
# 3. **WLL 대응**: `scale_pos_weight=1.0` + **Isotonic Calibration**
# 4. **AP 최적화**: **OOF 기반 Rank 보정**
# 5. **Model Assessment**: CV로 **test error 추정**

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# 디렉토리 설정
notebook_dir = r'C:\Users\tkdwl\Desktop\토스 경진대회'
os.chdir(notebook_dir)

# 데이터 로드
df_train = pd.read_parquet('./train.parquet')
df_test = pd.read_parquet('./test.parquet')

print(f"Train: {df_train.shape} | Test: {df_test.shape}")

# ## 1. 결측 처리 (슬라이드: Resampling 전 데이터 정제)

target_col = 'clicked'
id_col = 'ID'
col_drop_threshold = 0.05

# 5% 이상 결측 제거
missing = pd.concat([df_train.isnull().mean(), df_test.isnull().mean()], axis=1).max(axis=1)
high_missing_cols = missing[missing >= col_drop_threshold].index.tolist()
high_missing_cols = [c for c in high_missing_cols if c not in [target_col, id_col]]

df_train = df_train.drop(columns=high_missing_cols)
df_test = df_test.drop(columns=high_missing_cols)

# 수치형/범주형 분리
num_cols = df_train.select_dtypes(include=np.number).columns.tolist()
num_cols = [c for c in num_cols if c != target_col]
cat_cols = df_train.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c not in [id_col]]

# 수치형 결측 행 제거
df_train = df_train.dropna(subset=num_cols)
df_test = df_test.dropna(subset=num_cols)

# 범주형 최빈값
from sklearn.impute import SimpleImputer
cat_imputer = SimpleImputer(strategy='most_frequent')
df_train[cat_cols] = cat_imputer.fit_transform(df_train[cat_cols])
df_test[cat_cols] = cat_imputer.transform(df_test[cat_cols])

print(f"결측 처리 완료")

# ## 2. 범주형 p-value 필터링 (Classification 슬라이드)

from sklearn.preprocessing import OneHotEncoder
import statsmodels.api as sm

# day_of_week를 사용하여 is_weekend 피처 먼저 생성 (삭제 전)
if 'day_of_week' in df_train.columns:
    df_train['is_weekend'] = df_train['day_of_week'].astype(str).str.contains('5|6', regex=True).astype(int)
    df_test['is_weekend'] = df_test['day_of_week'].astype(str).str.contains('5|6', regex=True).astype(int)
    # Optionally drop 'day_of_week' if not needed
    # df_train.drop('day_of_week', axis=1, inplace=True)
    # df_test.drop('day_of_week', axis=1, inplace=True)

selected_cats = []
y = df_train[target_col]

for cat in cat_cols:
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    X_cat = encoder.fit_transform(df_train[[cat]])
    X_cat = sm.add_constant(X_cat)
    model = sm.Logit(y, X_cat).fit_regularized(method='l1', alpha=1e-5, maxiter=100, trim_mode='size', disp=False)
    pvals = model.pvalues[1:]  # constant 제외
    significant = (pvals < 0.05).sum() > 0
    if significant:
        selected_cats.append(cat)

print(f"선택된 범주형 피처: {len(selected_cats)}개")

# 고차원 범주형 제거 (if not already handled)
cardinality = df_train[cat_cols].nunique()
high_card_cols = cardinality[cardinality > 50].index.tolist()
cat_cols = [c for c in cat_cols if c not in high_card_cols]

# seq 피처
def process_seq(df, top_n=15):
    df['seq_length'] = df['seq'].str.split(',').str.len().fillna(0)
    df['seq_unique'] = df['seq'].str.split(',').apply(lambda x: len(set(x)) if isinstance(x, str) else 0)
    df['seq_diversity'] = df['seq_unique'] / (df['seq_length'] + 1e-8)
    items = [i.strip() for s in df['seq'].str.split(',') for i in s if isinstance(s, str)]
    top_items = pd.Series(items).value_counts().head(50).index
    for item in top_items[:top_n]:
        df[f'seq_has_{item}'] = df['seq'].str.contains(item, regex=False).fillna(0).astype(int)
    return df

df_train = process_seq(df_train)
df_test = process_seq(df_test)

# 클러스터링
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import gc

cluster_cols = [c for c in num_cols if c in df_train.columns]
scaler = StandardScaler()
scaled_train = scaler.fit_transform(df_train[cluster_cols])
scaled_test = scaler.transform(df_test[cluster_cols])

kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
df_train['cluster'] = kmeans.fit_predict(scaled_train)
df_test['cluster'] = kmeans.predict(scaled_test)

df_train = pd.get_dummies(df_train, columns=['cluster'], prefix='cluster')
df_test = pd.get_dummies(df_test, columns=['cluster'], prefix='cluster')
df_test = df_test.reindex(columns=df_train.columns, fill_value=0)

# 수치형 스케일링
num_cols_final = [c for c in num_cols if c in df_train.columns]
df_train[num_cols_final] = scaler.fit_transform(df_train[num_cols_final])
df_test[num_cols_final] = scaler.transform(df_test[num_cols_final])

print(f"최종 피처 수: {df_train.shape[1] - 2}")
gc.collect()

df_train.to_parquet('./processed_data/processed_train.parquet', index=False)
df_test.to_parquet('./processed_data/processed_test.parquet', index=False)
print(f"처리된 데이터 저장 완료: processed_data/")




# ## 4. 5-fold CV + LightGBM

import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score, log_loss
from sklearn.isotonic import IsotonicRegression
import gc

feature_cols = [c for c in df_train.columns if c not in [target_col, id_col]]
X = df_train[feature_cols]
y = df_train[target_col]

# WLL: 50:50 가중 → scale_pos_weight=1.0
params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.03,
    'num_leaves': 128,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'verbose': -1,
    'seed': 42
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(df_test))
ap_scores = []

for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y)):
    X_trn, X_val = X.iloc[trn_idx], X.iloc[val_idx]
    y_trn, y_val = y.iloc[trn_idx], y.iloc[val_idx]

    lgb_train = lgb.Dataset(X_trn, y_trn)
    lgb_valid = lgb.Dataset(X_val, y_val, reference=lgb_train)

    model = lgb.train(
        params,
        lgb_train,
        num_boost_round=10000,
        valid_sets=[lgb_valid],
        early_stopping_rounds=300,
        verbose_eval=1000
    )

    val_pred = model.predict(X_val)
    oof_preds[val_idx] = val_pred

    # AP 계산
    ap = average_precision_score(y_val, val_pred)
    ap_scores.append(ap)
    print(f"Fold {fold+1} AP: {ap:.5f}")

    test_preds += model.predict(df_test[feature_cols]) / skf.n_splits

print(f"\nCV AP: {np.mean(ap_scores):.5f} ± {np.std(ap_scores):.5f}")
gc.collect()

# Additional steps like Isotonic Calibration and submission can be added here if needed.